In [1]:
!pip install einops

In [ ]:
import torch
from einops import rearrange

In [15]:
# Create a 4D tensor of shape (batch, channels, height, width)
tensor = torch.rand(10, 3, 32, 32)  # Example: a batch of 10 RGB images 32x32
# Rearrange to (batch, height, width, channels) for image processing libraries that expect this format
rearranged = rearrange(tensor, 'x1 x2 x3 x4 -> x4 x3 x2 x1')
print(rearranged.size()) 

torch.Size([32, 32, 3, 10])


In [16]:
from einops import reduce

# Reduce the tensor's channel dimension by taking the mean, resulting in a grayscale image
grayscale = reduce(tensor, 'x1 x2 x3 x4 -> x2 x3 x4', 'mean')
print(grayscale.size())

torch.Size([3, 32, 32])


In [17]:
from einops import repeat

# Repeat each image in the batch 4 times along a new dimension
repeated = repeat(tensor, 'x1 x2 x3 x4 -> (repeat x1) x2 x3 x4', repeat=4)
print(repeated.size())

torch.Size([40, 3, 32, 32])


In [19]:
# Split channels
red, green, blue = rearrange(tensor, 'b (c rgb) h w -> rgb b c h w', rgb=3)

# Example processing (identity here)
processed_red, processed_green, processed_blue = red, green, blue
print(processed_red.size(), processed_green.size(), processed_blue.size())
      
# Merge channels back
merged = rearrange([processed_red, processed_green, processed_blue], 'rgb b c h w -> b (rgb c) h w')
print(merged.size())

torch.Size([10, 1, 32, 32]) torch.Size([10, 1, 32, 32]) torch.Size([10, 1, 32, 32])
torch.Size([10, 3, 32, 32])


In [20]:
# Flatten spatial dimensions
flattened = rearrange(tensor, 'b x1 x2 x3 -> b (x1 x2 x3)')
print(flattened.size())

# Example neural network operation
# output = model(flattened)
# Unflatten back to spatial dimensions (assuming output has shape b, features)
# unflattened = rearrange(output, 'b (c h w) -> b c h w', c=3, h=32, w=32)

torch.Size([10, 3072])


In [24]:
# Assuming tensor is batch of images b, c, h, w

# Batch of images: batch, channels, height, width
tensor = torch.rand(10, 3, 32, 32)
crop_size = 24
start = (tensor.shape[-1] - crop_size) // 2
cropped = tensor[
    :,
    :,
    start:start + crop_size,
    start:start + crop_size
]

# Example: BCHW -> BHWC
cropped = rearrange(cropped, 'b c h w -> b h w c')
print(cropped.size())

torch.Size([10, 24, 24, 3])


### Simplified Self-Attention

In [25]:
import torch.nn.functional as F

def simplified_self_attention(q, k, v):
    """
    A simplified self-attention mechanism.
    Args:
        q, k, v (torch.Tensor): Queries, Keys, and Values. Shape: [batch_size, num_tokens, feature_dim]
    Returns:
        torch.Tensor: The result of the attention mechanism.
    """
    # Compute the dot product between queries and keys
    scores = torch.matmul(q, k.transpose(-2, -1))
    
    # Apply softmax to get probabilities
    attn_weights = F.softmax(scores, dim=-1)
    
    # Multiply by values
    output = torch.matmul(attn_weights, v)
    return output
# Example tensors representing queries, keys, and values
batch_size, num_tokens, feature_dim = 10, 16, 64
q = torch.rand(batch_size, num_tokens, feature_dim)
k = torch.rand(batch_size, num_tokens, feature_dim)
v = torch.rand(batch_size, num_tokens, feature_dim)
# Apply self-attention
attention_output = simplified_self_attention(q, k, v)
print("Output shape:", attention_output.shape)

Output shape: torch.Size([10, 16, 64])


### Multi-head Attention

In [26]:
def multi_head_self_attention(q, k, v, num_heads=8):
    """
    Multi-head self-attention using Einops for splitting and merging heads.
    """
    batch_size, num_tokens, feature_dim = q.shape
    head_dim = feature_dim // num_heads
    
    # Split into multiple heads
    q, k, v = [
        rearrange(x, 'b t (h d) -> b h t d', h=num_heads)
        for x in (q, k, v)
    ]
    
    # Apply self-attention to each head
    output = simplified_self_attention(q, k, v)
    
    # Merge the heads back
    output = rearrange(output, 'b h t d -> b t (h d)')
    return output

# Apply multi-head self-attention
multi_head_attention_output = multi_head_self_attention(q, k, v)
print("Multi-head output shape:", multi_head_attention_output.shape)

Multi-head output shape: torch.Size([10, 16, 64])


In [28]:
from einops import einsum

x = torch.ones(2, 3, 4)  # batch seq1 hidden
y = torch.ones(2, 3, 4)  # batch seq2 hidden

# Old way
z = x @ y.transpose(-2, -1)  # batch seq1 seq2

# New (einops) way
z = einsum(
    x,
    y,
    "batch seq1 hidden, batch seq2 hidden -> batch seq1 seq2",
)

# Or use `...` to represent broadcasting over any number of dimensions
z = einsum(
    x,
    y,
    "... seq1 hidden, ... seq2 hidden -> ... seq1 seq2",
)
print(z.size())

torch.Size([2, 3, 3])
